# 02 - Préparation des données

UCI HAR est déjà découpé en fenêtres (128 points, 2,56 s, 50 Hz), donc pas de fenêtrage à refaire. Ce notebook fait trois choses : empiler les 9 signaux bruts en un tableau par fenêtre, séparer une partie du train pour la validation par sujet, et calculer la normalisation uniquement sur les données d'entraînement réelles, pour la réutiliser à l'identique plus tard.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data/raw/UCI HAR Dataset")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

# Fixed channel order: every array we build, and later the simulation script,
# must stack the 9 raw signals in this exact same order.
CHANNELS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]
WINDOW_LENGTH = 128
SAMPLING_RATE_HZ = 50

## Charger et empiler les signaux

Chaque fichier `<signal>_<split>.txt` contient une ligne par fenêtre. On charge les 9 fichiers et on les empile sur un nouvel axe : le résultat a la forme `(nombre de fenêtres, 128, 9)`, exactement la forme qu'un modèle `Conv1D` attend (temps, capteurs).

In [2]:
def load_ids(path):
    return pd.read_csv(path, header=None, names=["value"]).squeeze("columns")

def load_signal(split, signal_name):
    path = DATA_DIR / split / "Inertial Signals" / f"{signal_name}_{split}.txt"
    return np.loadtxt(path)

def load_windows(split):
    signals = [load_signal(split, name) for name in CHANNELS]
    return np.stack(signals, axis=-1)  # (n_windows, 128, 9)

X_train_full = load_windows("train")
X_test = load_windows("test")

subject_train_full = load_ids(DATA_DIR / "train" / "subject_train.txt")
subject_test = load_ids(DATA_DIR / "test" / "subject_test.txt")

y_train_full = load_ids(DATA_DIR / "train" / "y_train.txt") - 1  # 0-indexed labels for the model
y_test = load_ids(DATA_DIR / "test" / "y_test.txt") - 1

print("X_train_full:", X_train_full.shape)
print("X_test:     ", X_test.shape)

X_train_full: (7352, 128, 9)
X_test:      (2947, 128, 9)


## Labels

On sauvegarde tout de suite la correspondance entier -> nom d'activité, dans le même ordre que `activity_labels.txt`, décalée de 1 pour commencer à 0 (ce que Keras attend pour la classification).

In [3]:
activity_labels = pd.read_csv(DATA_DIR / "activity_labels.txt", sep=r"\s+", header=None, names=["id", "activity"])
labels_map = {str(row.id - 1): row.activity for row in activity_labels.itertuples()}

with open(MODELS_DIR / "labels.json", "w") as f:
    json.dump(labels_map, f, indent=2)

labels_map

{'0': 'WALKING',
 '1': 'WALKING_UPSTAIRS',
 '2': 'WALKING_DOWNSTAIRS',
 '3': 'SITTING',
 '4': 'STANDING',
 '5': 'LAYING'}

## Split entraînement / validation, par sujet

Même logique que pour train/test : des fenêtres qui se chevauchent et appartiennent à la même personne ne doivent pas se retrouver à la fois dans le fit et la validation, sinon la mesure serait faussée. On retire donc des sujets entiers, jamais des fenêtres isolées.

On garde environ 20 % des 21 sujets du train pour la validation, avec une `random_state` fixe pour que ce choix soit reproductible.

In [4]:
unique_train_subjects = sorted(subject_train_full.unique())

fit_subjects, val_subjects = train_test_split(
    unique_train_subjects, test_size=0.2, random_state=42
)

fit_mask = subject_train_full.isin(fit_subjects).to_numpy()
val_mask = subject_train_full.isin(val_subjects).to_numpy()

X_fit, y_fit = X_train_full[fit_mask], y_train_full[fit_mask].to_numpy()
X_val, y_val = X_train_full[val_mask], y_train_full[val_mask].to_numpy()

print("fit subjects:", fit_subjects)
print("validation subjects:", val_subjects)
print("X_fit:", X_fit.shape, " X_val:", X_val.shape)

fit subjects: [np.int64(8), np.int64(19), np.int64(6), np.int64(28), np.int64(26), np.int64(22), np.int64(5), np.int64(16), np.int64(30), np.int64(7), np.int64(21), np.int64(14), np.int64(17), np.int64(23), np.int64(29), np.int64(11)]
validation subjects: [np.int64(1), np.int64(27), np.int64(25), np.int64(3), np.int64(15)]
X_fit: (5551, 128, 9)  X_val: (1801, 128, 9)


## Normalisation

Moyenne et écart-type calculés uniquement sur `X_fit` (les vraies données d'entraînement, sujets de validation exclus), jamais sur la validation ni le test. Une valeur par capteur, calculée sur toutes les fenêtres et tous les instants à la fois.

In [5]:
flat_fit = X_fit.reshape(-1, X_fit.shape[-1])  # (n_fit_windows * 128, 9)
mean = flat_fit.mean(axis=0)
std = flat_fit.std(axis=0)

def normalize(X):
    return (X - mean) / std

X_fit_norm = normalize(X_fit)
X_val_norm = normalize(X_val)
X_test_norm = normalize(X_test)

for name, arr in [("fit", X_fit_norm), ("val", X_val_norm), ("test", X_test_norm)]:
    print(f"{name}: mean~{arr.mean():.3f} std~{arr.std():.3f} shape={arr.shape}")

fit: mean~0.000 std~1.000 shape=(5551, 128, 9)
val: mean~0.017 std~0.924 shape=(1801, 128, 9)
test: mean~-0.003 std~0.938 shape=(2947, 128, 9)


In [6]:
normalization = {
    "channels": CHANNELS,
    "mean": mean.tolist(),
    "std": std.tolist(),
    "window_length": WINDOW_LENGTH,
    "sampling_rate_hz": SAMPLING_RATE_HZ,
    "validation_subjects": [int(s) for s in val_subjects],
}

with open(MODELS_DIR / "normalization.json", "w") as f:
    json.dump(normalization, f, indent=2)

normalization

{'channels': ['body_acc_x',
  'body_acc_y',
  'body_acc_z',
  'body_gyro_x',
  'body_gyro_y',
  'body_gyro_z',
  'total_acc_x',
  'total_acc_y',
  'total_acc_z'],
 'mean': [-0.0006061088322023841,
  -0.00027456262208354603,
  -0.00023890472153037648,
  0.0005248877977051498,
  -0.0009286430427825374,
  -0.0010742662324096381,
  0.7996139359543786,
  0.027382570872351297,
  0.08024868637816124],
 'std': [0.1991436376612025,
  0.12222116238214013,
  0.11046840207172769,
  0.40762428707908926,
  0.39307714650416736,
  0.26698869660891994,
  0.4193434776385132,
  0.3835483034246538,
  0.3754756222268382],
 'window_length': 128,
 'sampling_rate_hz': 50,
 'validation_subjects': [1, 27, 25, 3, 15]}

## Observations

- **Tailles :** 5551 fenêtres fit, 1801 validation (sujets 1, 27, 25, 3, 15), 2947 test. Aucun sujet en commun.
- **Normalisation :** sur `X_fit`, moyenne ~0 et écart-type ~1 par construction. Sur validation (0.017 / 0.924) et test (-0.003 / 0.938), les valeurs restent proches de 0/1, logique puisque ces personnes n'ont pas servi au calcul.
- **`total_acc_x`** a une moyenne brute ~0.80 avant normalisation : c'est la gravité (~1g), retirée par les auteurs dans les signaux `body_acc_*` qui restent proches de 0.
- `labels.json` et `normalization.json` sont dans `models/` : le script de simulation les rechargera pour prétraiter les données exactement pareil qu'à l'entraînement.